In [ ]:
# =============================================================================
# FDA NDC Directory Load
# =============================================================================
# Downloads the current openFDA National Drug Code (NDC) product file:
# https://download.open.fda.gov/drug/ndc/drug-ndc-0001-of-0001.json.zip
#
# Saves the zip to the configured out_folder.
#
# The file contains the full FDA NDC Directory — one record per marketed
# drug product, including product NDC, package NDC, brand/generic name,
# labeler, dosage form, route, active ingredients, strength, and listing status.
#
# This script flattens the nested JSON, normalizes package NDC to standard
# NDC-11 format, and flags whether a package NDC is available.
# =============================================================================

import pandas as pd
import zipfile
import json
import requests
from pathlib import Path
from config import out_folder

# Download location
zip_path = Path(out_folder) / "drug-ndc-0001-of-0001.json.zip"
url = "https://download.open.fda.gov/drug/ndc/drug-ndc-0001-of-0001.json.zip"

print("Downloading FDA NDC file...")
response = requests.get(url, stream=True)
response.raise_for_status()

with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"Downloaded to: {zip_path}")

# Open the zip and load the JSON
with zipfile.ZipFile(zip_path) as z:
    json_filename = z.namelist()[0]
    with z.open(json_filename) as f:
        data = json.load(f)

results = data.get("results", [])
print(f"Total records in file: {len(results):,}")

# Flatten the records
records = []
for item in results:
    ingredients = item.get("active_ingredients") or []
    first_ingredient = ingredients[0] if ingredients else {}

    packaging = item.get("packaging") or []
    first_package = packaging[0] if packaging else {}

    flat = {
        "product_ndc": item.get("product_ndc"),
        "generic_name": item.get("generic_name"),
        "brand_name": item.get("brand_name"),
        "labeler_name": item.get("labeler_name"),
        "dosage_form": item.get("dosage_form"),
        "route": ", ".join(item.get("route") or []),
        "marketing_category": item.get("marketing_category"),
        "listing_expiration_date": item.get("listing_expiration_date"),
        "product_type": item.get("product_type"),
        "finished": item.get("finished"),
        "active_ingredient": first_ingredient.get("name"),
        "strength": first_ingredient.get("strength"),
        "package_ndc": first_package.get("package_ndc"),
        "rxcui": ", ".join(item.get("openfda", {}).get("rxcui") or []),
    }
    records.append(flat)

df = pd.DataFrame(records)
print(f"Final row count: {len(df):,}")

# Schema check
required_cols = {
    "product_ndc", "generic_name", "brand_name", "labeler_name",
    "dosage_form", "route", "marketing_category", "listing_expiration_date",
    "product_type", "finished", "active_ingredient", "strength",
    "package_ndc", "rxcui"
}

missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"FDA schema changed — missing columns: {sorted(missing)}")

print("FDA schema OK — all expected columns present.")
print(f"Null rates:\n{df.isna().mean().sort_values(ascending=False).head(8)}")

def to_ndc11(ndc, hyphenated=False):
    if not ndc or not isinstance(ndc, str):
        return None
    ndc = ndc.strip()
    parts = ndc.split("-")
    if len(parts) == 1 and len(ndc) == 11 and ndc.isdigit():
        parts = [ndc[:5], ndc[5:9], ndc[9:]]
    elif len(parts) != 3 or not all(p.isdigit() for p in parts):
        return None
    lengths = tuple(len(p) for p in parts)
    if lengths not in {(4, 4, 2), (5, 3, 2), (5, 4, 1), (5, 4, 2)}:
        return None
    labeler = parts[0].zfill(5)
    product = parts[1].zfill(4)
    package = parts[2].zfill(2)
    if hyphenated:
        return f"{labeler}-{product}-{package}"
    return labeler + product + package

df["ndc11"] = df["package_ndc"].apply(lambda x: to_ndc11(x) if pd.notna(x) else None)
df["package_ndc_status"] = df["package_ndc"].apply(
    lambda x: "Available" if pd.notna(x) else "Package not available"
)

print(df[["package_ndc", "ndc11", "package_ndc_status"]].head(10))
print(f"Rows with valid ndc11: {df['ndc11'].notna().sum():,}")

In [ ]:
#Run to expand for dataset
pd.set_option("display.max_colwidth", 40)
df.head(1)

In [ ]:
# =============================================================================
# RxNorm NDC Crosswalk (from zip)
# =============================================================================
# Dynamically finds and downloads the latest RxNorm Full Prescribe file
# from the NLM RxNorm Files page:
# https://www.nlm.nih.gov/research/umls/rxnorm/docs/rxnormfiles.html
#
# Saves the zip to the configured out_folder.
#
# The RxNorm Full Prescribe file is a monthly release that contains the
# full set of normalized drug concepts (RxCUI) and their relationships,
# including NDC mappings. This script extracts the NDC-to-RxCUI crosswalk
# from the RXNSAT.RRF file (attribute records where ATN = 'NDC' and
# SAB = 'RXNORM').
#
# Output is a clean one-to-many mapping of NDC-11 to RxCUI.
# =============================================================================

import pandas as pd
import zipfile
import requests
from pathlib import Path
from bs4 import BeautifulSoup

# Dynamically find the latest RxNorm full prescribe zip URL
page_url = "https://www.nlm.nih.gov/research/umls/rxnorm/docs/rxnormfiles.html"
resp = requests.get(page_url)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

link = soup.find("a", href=lambda href: href and "RxNorm_full_prescribe_" in href and href.endswith(".zip"))
if not link:
    raise ValueError("Could not find latest RxNorm prescribe zip link on the page.")

url = link["href"]
zip_name = url.split("/")[-1]
zip_path = Path(zip_name)

print(f"Latest URL: {url}")

if not zip_path.exists():
    print("Downloading...")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print("Download complete.")

with zipfile.ZipFile(zip_path) as z:
    sat = pd.read_csv(
        z.open("rrf/RXNSAT.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 8, 9, 10],
        names=["rxcui", "atn", "sab", "ndc11"]
    )

ndc_xwalk = (
    sat[(sat["atn"] == "NDC") & (sat["sab"] == "RXNORM")]
    [["ndc11", "rxcui"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(ndc_xwalk.shape)
print(ndc_xwalk.head())
print("Unique NDCs:", ndc_xwalk["ndc11"].nunique())
print("Unique RxCUIs:", ndc_xwalk["rxcui"].nunique())

In [ ]:
# =============================================================================
# RxNorm Concepts (from local zip, no API)
# =============================================================================
# Reads the previously downloaded RxNorm Full Prescribe zip
# (RxNorm_full_prescribe_*.zip) from the configured out_folder.
#
# Extracts preferred concept names from the RXNCONSO.RRF file.
# RXNCONSO contains one row per term for each RxCUI. This script keeps
# only the most useful term types (SCD, SBD, GPCK, BPCK, IN, PIN, MIN, BN)
# and returns one clean record per RxCUI with:
#   - name      (preferred term)
#   - tty       (term type)
#   - suppress  (RxNorm suppression status)
#
# Used to enrich the NDC crosswalk with human-readable concept names
# without calling the RxNav API.
# =============================================================================

import pandas as pd
import zipfile
from pathlib import Path
from config import out_folder

# Reuse the same zip you already downloaded
zip_name = [f for f in Path(out_folder).glob("RxNorm_full_prescribe_*.zip")][0]
print(f"Using: {zip_name.name}")

with zipfile.ZipFile(zip_name) as z:
    conso = pd.read_csv(
        z.open("rrf/RXNCONSO.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 11, 12, 13, 14],
        names=["rxcui", "suppress", "tty", "code", "str"]
    )

# Keep only the preferred RxNorm term for each RxCUI
concepts = (
    conso[conso["tty"].isin(["SCD", "SBD", "GPCK", "BPCK", "IN", "PIN", "MIN", "BN"])]
    .drop_duplicates(subset="rxcui")
    [["rxcui", "str", "tty", "suppress"]]
    .rename(columns={"str": "name"})
    .reset_index(drop=True)
)

print(concepts.shape)
print(concepts.head())
print("Unique RxCUIs:", concepts["rxcui"].nunique())

In [ ]:
# =============================================================================
# Join All Three (FDA NDC + RxNorm Crosswalk + RxNorm Concepts)
# =============================================================================
# Combines the three prepared datasets:
#
#   1. FDA NDC Directory (df)          — product and package-level NDC data
#   2. RxNorm NDC Crosswalk (ndc_xwalk) — NDC-11 → RxCUI mapping
#   3. RxNorm Concepts (concepts)       — RxCUI → preferred name, TTY, suppress
#
# Join path:
#   FDA.ndc11  →  Crosswalk.ndc11   (left join)
#   Crosswalk.rxcui  →  Concepts.rxcui  (left join)
#
# Result is one row per FDA package NDC with RxNorm concept information
# attached where a match exists. Unmatched NDCs are retained with null
# RxNorm columns.
# =============================================================================

joined = (
    df.merge(ndc_xwalk, on="ndc11", how="left", suffixes=("", "_xwalk"))
      .merge(concepts, left_on="rxcui_xwalk", right_on="rxcui", how="left", suffixes=("", "_concept"))
)

print(f"Joined shape: {joined.shape}")
print(f"Matched to RxNorm crosswalk: {joined['rxcui_xwalk'].notna().sum():,}")
print(f"Matched to concept name: {joined['name'].notna().sum():,}")
joined.head(1)

In [ ]:
# =============================================================================
# RxNorm TTY Lookup
# =============================================================================
# Adds a human-readable description for each RxNorm Term Type (TTY).
#
# TTY is the term type code from RxNorm (e.g., SCD, SBD, IN, BN).
# This lookup translates the code into its full name so the joined file
# is easier to read without having to reference the RxNorm documentation.
#
# Common values in this dataset:
#   SCD  = Semantic Clinical Drug
#   SBD  = Semantic Branded Drug
#   IN   = Ingredient
#   BN   = Brand Name
#   BPCK = Branded Pack
#   GPCK = Generic Pack
#
# The lookup is left-joined to the final dataset on the `tty` column.
# =============================================================================

tty_lookup = pd.DataFrame({
    "tty": [
        "SCD", "SBD", "GPCK", "BPCK",
        "IN", "PIN", "BN", "DF",
        "SCDF", "SBDF", "SCDG", "SBDG",
        "MIN", "TMSY", "SY", "PSN"
    ],
    "term_type": [
        "Semantic Clinical Drug",
        "Semantic Branded Drug",
        "Generic Pack",
        "Branded Pack",
        "Ingredient",
        "Precise Ingredient",
        "Brand Name",
        "Dose Form",
        "Semantic Clinical Drug Form",
        "Semantic Branded Drug Form",
        "Semantic Clinical Drug Group",
        "Semantic Branded Drug Group",
        "Multiple Ingredients",
        "Tall Man Synonym",
        "Synonym",
        "Prescribable Name"
    ]
})

# Attach meaning to the joined data
joined = joined.merge(tty_lookup, on="tty", how="left")

print(f"TTY values mapped: {joined['term_type'].notna().sum():,}")
print("\nSample TTY breakdown:")
print(joined["term_type"].value_counts(dropna=False).head(1))

In [ ]:
# =============================================================================
# Final Join — FDA NDC + RxNorm
# -----------------------------------------------------------------------------
# Combined dataset sources:
#   1. FDA NDC Directory
#   2. RxNorm NDC Crosswalk
#   3. RxNav Concepts
#
# Output: cleaned joined dataframe (final)
# =============================================================================

# Use the FDA + RxNorm join result
final = joined.copy()   # change to whatever your clean FDA+RxNorm dataframe is named

# Optional: drop any leftover suffix columns if present
final = final.loc[:, ~final.columns.str.contains(r"_xwalk$|_concept$", regex=True)]

print(f"Final shape: {final.shape}")
print("Columns:", final.columns.tolist()[:15])
final.head(1)

# =============================================================================
# Final Export — Excel with auto-fit columns
# =============================================================================

from pathlib import Path
from config import out_folder
import pandas as pd

out_folder = Path(out_folder)
out_folder.mkdir(parents=True, exist_ok=True)

excel_path = out_folder / "fda_rxnorm_joined.xlsx"

# Compute widths from the dataframe (faster than cell-by-cell)
col_widths = {}
for col in final.columns:
    max_len = len(str(col))
    sample = final[col].head(500).astype(str)
    if len(sample) > 0:
        max_len = max(max_len, sample.str.len().max())
    col_widths[col] = min(max_len + 2, 69)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    final.to_excel(writer, index=False, sheet_name="joined")
    
    worksheet = writer.sheets["joined"]
    for i, col in enumerate(final.columns, 1):
        worksheet.column_dimensions[worksheet.cell(1, i).column_letter].width = col_widths[col]

print(f"Excel exported: {excel_path}")
print(f"Final shape: {final.shape}")